Question 1: Calculate Average User Session Duration

You have a dataset containing user activity logs in a PySpark DataFrame with the following columns:
 user_id, timestamp, and action. 

The action column indicates whether the userstarted or ended a session. It can have values 'start' or 'end'. Your task is to calculate the average duration of user sessions.

user_id | timestamp          | action
------- | ------------------ | ------
1       | 2022-01-01 10:00   | start
1       | 2022-01-01 10:15   | end
2       | 2022-01-01 11:00   | start
1       | 2022-01-01 11:30   | start
2       | 2022-01-01 11:45   | end
1       | 2022-01-01 12:00   | end

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, unix_timestamp, avg, when

# Initialize Spark Session
spark = SparkSession.builder.appName("AverageSessionDuration").getOrCreate()

# Sample Data
data = [
    (1, "2022-01-01 10:00", "start"),
    (1, "2022-01-01 10:15", "end"),
    (2, "2022-01-01 11:00", "start"),
    (1, "2022-01-01 11:30", "start"),
    (2, "2022-01-01 11:45", "end"),
    (1, "2022-01-01 12:00", "end"),
]
columns = ["user_id", "timestamp", "action"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Convert timestamp to proper TimestampType
df = df.withColumn("timestamp", col("timestamp").cast("timestamp"))

# Define window partitioned by user_id and ordered by timestamp
window_spec = Window.partitionBy("user_id").orderBy("timestamp")

# Pair start and end actions: Use lag() to find the previous action and timestamp
df_with_lag = df.withColumn("prev_action", lag("action").over(window_spec)) \
                .withColumn("prev_timestamp", lag("timestamp").over(window_spec))

# Filter for valid session pairs (start -> end)
valid_sessions = df_with_lag.filter((col("action") == "end") & (col("prev_action") == "start"))

# Calculate session duration in seconds
valid_sessions = valid_sessions.withColumn(
    "session_duration", 
    unix_timestamp("timestamp") - unix_timestamp("prev_timestamp")
)

# Show individual session durations
valid_sessions.select("user_id", "prev_timestamp", "timestamp", "session_duration").show()

# Calculate average session duration
average_session_duration = valid_sessions.groupBy("user_id") \
                                         .agg(avg("session_duration").alias("avg_session_duration"))

# Show the average session duration per user
average_session_duration.show()

# Calculate overall average session duration across all users
overall_avg_duration = valid_sessions.agg(avg("session_duration").alias("overall_avg_session_duration"))
overall_avg_duration.show()


+-------+-------------------+-------------------+----------------+
|user_id|     prev_timestamp|          timestamp|session_duration|
+-------+-------------------+-------------------+----------------+
|      1|2022-01-01 10:00:00|2022-01-01 10:15:00|             900|
|      1|2022-01-01 11:30:00|2022-01-01 12:00:00|            1800|
|      2|2022-01-01 11:00:00|2022-01-01 11:45:00|            2700|
+-------+-------------------+-------------------+----------------+

+-------+--------------------+
|user_id|avg_session_duration|
+-------+--------------------+
|      1|              1350.0|
|      2|              2700.0|
+-------+--------------------+

+----------------------------+
|overall_avg_session_duration|
+----------------------------+
|                      1800.0|
+----------------------------+



Analyzing Sales Data

Problem Statement: You are given a CSV file containing sales data with the following columns: date, product_id, quantity_sold, and price_per_unit. Your task is to write a PySpark program to analyze this sales data and produce the following insights:
    
•Total revenue for each product.

•Total revenue for each date.

•Top 5 selling products based on the quantity sold.

date,product_id,quantity_sold,price_per_unit

2023-08-01,101,5,10.0

2023-08-01,102,3,15.0

2023-08-02,101,8,10.0

2023-08-02,103,6,12.0

2023-08-03,102,4,15.0

2023-08-03,104,2,20.0

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, desc

# Initialize Spark Session
spark = SparkSession.builder.appName("Sales Data Analysis").getOrCreate()

# Define the dataset
data = [
    ("2023-08-01", 101, 5, 10.0),
    ("2023-08-01", 102, 3, 15.0),
    ("2023-08-02", 101, 8, 10.0),
    ("2023-08-02", 103, 6, 12.0),
    ("2023-08-03", 102, 4, 15.0),
    ("2023-08-03", 104, 2, 20.0)
]
columns = ["date", "product_id", "quantity_sold", "price_per_unit"]

# Load the data into a DataFrame
sales_df = spark.createDataFrame(data, columns)

# Display the original data
print("Original Sales Data:")
sales_df.show()

# Step 1: Calculate Total Revenue for Each Product
product_revenue = sales_df.withColumn("revenue", col("quantity_sold") * col("price_per_unit")) \
                          .groupBy("product_id") \
                          .agg(sum("revenue").alias("total_revenue"))

print("Total Revenue for Each Product:")
product_revenue.show()

# Step 2: Calculate Total Revenue for Each Date
date_revenue = sales_df.withColumn("revenue", col("quantity_sold") * col("price_per_unit")) \
                       .groupBy("date") \
                       .agg(sum("revenue").alias("total_revenue"))

print("Total Revenue for Each Date:")
date_revenue.show()

# Step 3: Find Top 5 Selling Products Based on Quantity Sold
top_products = sales_df.groupBy("product_id") \
                       .agg(sum("quantity_sold").alias("total_quantity_sold")) \
                       .orderBy(desc("total_quantity_sold")) \
                       .limit(5)

print("Top 5 Selling Products Based on Quantity Sold:")
top_products.show()


Original Sales Data:
+----------+----------+-------------+--------------+
|      date|product_id|quantity_sold|price_per_unit|
+----------+----------+-------------+--------------+
|2023-08-01|       101|            5|          10.0|
|2023-08-01|       102|            3|          15.0|
|2023-08-02|       101|            8|          10.0|
|2023-08-02|       103|            6|          12.0|
|2023-08-03|       102|            4|          15.0|
|2023-08-03|       104|            2|          20.0|
+----------+----------+-------------+--------------+

Total Revenue for Each Product:
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|       101|        130.0|
|       102|        105.0|
|       103|         72.0|
|       104|         40.0|
+----------+-------------+

Total Revenue for Each Date:
+----------+-------------+
|      date|total_revenue|
+----------+-------------+
|2023-08-01|         95.0|
|2023-08-02|        152.0|
|2023-08-03|        100.0|
+-------